In [7]:
import cv2
import numpy as np
import os
from tensorflow.keras.models import load_model

# โหลดโมเดล
model = load_model(r'C:\Users\somchay\projuctworkshop\AIproject.keras')  # ใช้ raw string

# ค้นหาภาพทั้งหมดในโฟลเดอร์ Test
test_folder = r"C:\Users\somchay\projuctworkshop\Test"
test_images = []
for root, dirs, files in os.walk(test_folder):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            test_images.append(os.path.join(root, file))

# กำหนด Threshold
THRESHOLD = 0.7  # เปลี่ยนเป็น 0.7 สำหรับการเปรียบเทียบกับค่าความมั่นใจ

# ทำนายและแสดงผล
for img_path in test_images:
    image = cv2.imread(img_path)
    if image is None:
        print(f"Error: ไม่พบไฟล์ {img_path}")
        continue
    
    # แปลงสีและประมวลผลภาพ
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    resized_img = cv2.resize(image_rgb, (200, 200))  # ปรับขนาดให้ตรงกับโมเดล
    normalized_img = resized_img.astype("float32") / 255.0
    input_img = np.expand_dims(normalized_img, axis=0)
    
    # ทำนายผล
    prediction = model.predict(input_img, verbose=0)
    
    # สำหรับ binary classification
    normal_confidence = prediction[0][0] * 100  
    funny_confidence = (1 - prediction[0][0]) * 100  

    # กำหนด label
    if funny_confidence > THRESHOLD * 100:  
        label = f"Funny ({funny_confidence:.1f}%)"  
    elif normal_confidence > THRESHOLD * 100:
        label = f"Normal ({normal_confidence:.1f}%)"
    else:
        label = "Uncertain"

    # แสดงผล
    print(f"Image: {img_path}")
    print(f"Actual Class: {os.path.basename(os.path.dirname(img_path))}")
    print(f"Prediction: {label}")
    print("----------------------------------")


Image: C:\Users\somchay\projuctworkshop\Test\Funny Face\IMG_1649.jpg
Actual Class: Funny Face
Prediction: Funny (90.4%)
----------------------------------
Image: C:\Users\somchay\projuctworkshop\Test\Funny Face\IMG_1650.jpg
Actual Class: Funny Face
Prediction: Funny (99.2%)
----------------------------------
Image: C:\Users\somchay\projuctworkshop\Test\Funny Face\IMG_1651.jpg
Actual Class: Funny Face
Prediction: Funny (98.8%)
----------------------------------
Image: C:\Users\somchay\projuctworkshop\Test\Funny Face\IMG_1652.jpg
Actual Class: Funny Face
Prediction: Funny (99.1%)
----------------------------------
Image: C:\Users\somchay\projuctworkshop\Test\Funny Face\IMG_1653.jpg
Actual Class: Funny Face
Prediction: Funny (98.8%)
----------------------------------
Image: C:\Users\somchay\projuctworkshop\Test\Funny Face\IMG_1654.jpg
Actual Class: Funny Face
Prediction: Funny (99.8%)
----------------------------------
Image: C:\Users\somchay\projuctworkshop\Test\Funny Face\IMG_1655.jpg
A

In [5]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# โหลดโมเดล
model = load_model(r'C:\Users\somchay\projuctworkshop\AIproject.keras')

# กำหนด Threshold
THRESHOLD = 0.7

# เปิดกล้อง
cap = cv2.VideoCapture(0)

# โหลด Haar Cascade สำหรับการตรวจจับใบหน้า
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

while True:
    # จับภาพจากกล้อง
    ret, frame = cap.read()
    if not ret:
        print("Error: ไม่สามารถจับภาพจากกล้องได้")
        break

    # แปลงสีและประมวลผลภาพ
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)  
    faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5)

    for (x, y, w, h) in faces:
        # ตัดภาพใบหน้าเพื่อทำการพยากรณ์
        face_roi = frame[y:y + h, x:x + w]
        resized_img = cv2.resize(face_roi, (200, 200))  
        normalized_img = resized_img.astype("float32") / 255.0
        input_img = np.expand_dims(normalized_img, axis=0)

        # ทำนายผล
        prediction = model.predict(input_img, verbose=0)

        # คำนวณค่าความมั่นใจ
        funny_confidence = (1 - prediction[0][0]) * 100  
        normal_confidence = prediction[0][0] * 100  

        # กำหนด label และสี
        if funny_confidence > THRESHOLD * 100:
            label = f"Funny ({funny_confidence:.1f}%)"
            color = (0, 255, 0)  # หน้า funny สีเขียวอี๊
        elif normal_confidence > THRESHOLD * 100:
            label = f"Normal ({normal_confidence:.1f}%)"
            color = (0, 255, 255)  # หน้าnormal  สีเหลืองอึงงง
        else:
            label = "Uncertain"
            color = (0, 0, 255)  # หน้าที่ uncertainสีแดงจี๊ด จ๊าดดด

        # วาดกรอบสี่เหลี่ยมรอบใบหน้าด้วยสีที่กำหนด
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)

        # แสดงผลบนภาพ
        cv2.putText(frame, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    # แสดงผลภาพ
    cv2.imshow('Webcam', frame)

    # ออกจากลูปเมื่อกด 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ปิดกล้องและหน้าต่าง
cap.release()
cv2.destroyAllWindows()
